# 🧪 Step 1: Research & Data Modelling  
## Petrol Stations & EV Charging Points in Berlin

**PR Branch Name:** `petrol-stations-berlin-data-modelling`

This notebook documents **Step 1** of the Gas / Petrol Stations data layer for Berlin,


---

## Scope
- Petrol / fuel stations  
- EV charging stations  

---

## Objectives
- Identify and document data sources  
- Fetch raw POI data from OpenStreetMap  
- Select relevant columns for modelling  
- Draft a standardized schema  
- Plan transformation and enrichment steps  

---

## 1.1 Data Source Discovery

### Topic
**Petrol Stations & EV Charging Infrastructure in Berlin**

---

### Main Data Source
**OpenStreetMap (OSM)**

- Public, crowdsourced geospatial database  
- **Update frequency:** Continuous (near real-time)  
- **Data type:** Dynamic (API-based)  

#### Relevant OSM Tags
- `amenity=fuel` → Petrol / gas stations  
- `amenity=charging_station` → EV charging points  

#### Reason for Selection
- Broad coverage of petrol stations across Berlin  
- Includes coordinates, names, operators, opening hours, and amenities  
- Open license and easy to query programmatically  

---

### Optional / Enrichment Sources

#### Berlin Open Data Portal
- Official EV charging datasets  
- Static or semi-static data  

#### Wikidata
- Operator / brand enrichment  



In [1]:
import sys
print(sys.executable)


/opt/anaconda3/bin/python3.13


In [2]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version)


Python: /opt/anaconda3/bin/python3.13
Version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]


In [3]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

print("✅ imports working")

✅ imports working


In [4]:
ox.settings.use_cache = True
ox.settings.log_console = True


## 📍 Fetch Petrol Stations & EV Charging Points from OpenStreetMap

In this section, we query **OpenStreetMap** using the **OSMnx** library to retrieve:

- Petrol stations (`amenity=fuel`)
- EV charging stations (`amenity=charging_station`)

The query returns a **GeoDataFrame** containing:
- Geometry (location of each station)
- OSM metadata such as name, address, operator, opening hours, and amenities

This dataset serves as the **raw input layer** for further cleaning, modelling, and enrichment steps.


In [5]:
tags_fuel = {"amenity": "fuel"}
gdf_fuel = ox.features.features_from_place("Berlin, Germany", tags=tags_fuel)

print("Fuel stations fetched:", len(gdf_fuel))
gdf_fuel.head(3)


Fuel stations fetched: 289


geometry amenity brand brand:wikidata  \
element id                                                                 
node    16541597   POINT (13.3456 52.54631)    fuel  Aral        Q565734   
        26756830  POINT (13.32822 52.53069)    fuel  Aral        Q565734   
        26867411   POINT (13.2945 52.50197)    fuel   NaN            NaN   

                 brand:wikipedia compressed_air fuel:HGV_diesel fuel:diesel  \
element id                                                                    
node    16541597      en:Aral AG            yes          retail         yes   
        26756830         de:Aral            yes             NaN         yes   
        26867411             NaN            yes             NaN         yes   

                 fuel:e10 fuel:octane_102  ... landuse  \
element id                                 ...           
node    16541597      yes          retail  ...     NaN   
        26756830      yes             yes  ...     NaN   
        26867411      yes             NaN  ...     NaN   

                 service:vehicle:car_repair service:vehicle:inspection  \
element id                                                               
node    16541597                        NaN                        NaN   
        26756830                        NaN                        NaN   
        26867411                        NaN                        NaN   

                 fuel:octane_92 height wikimedia_commons payment:H2_Delivery  \
element id                                                                     
node    16541597            NaN    NaN               NaN                 NaN   
        26756830            NaN    NaN               NaN                 NaN   
        26867411            NaN    NaN               NaN                 NaN   

                 payment:HyLane payment:ryd_pay start_date:fuel:h35  
element id                                                           
node    16541597            NaN             NaN                 NaN  
        26756830            NaN             NaN                 NaN  
        26867411            NaN             NaN                 NaN  

[3 rows x 145 columns]

In [6]:
tags_ev = {"amenity": "charging_station"}
gdf_ev = ox.features.features_from_place("Berlin, Germany", tags=tags_ev)

print("Charging stations fetched:", len(gdf_ev))
gdf_ev.head(3)

Charging stations fetched: 1555


geometry           amenity amperage  \
element id                                                                
node    425089052   POINT (13.3227 52.51369)  charging_station       32   
        429740871  POINT (13.38087 52.52366)  charging_station       32   
        429951629   POINT (13.32897 52.4994)  charging_station       32   

                  authentication:short_message bicycle capacity  \
element id                                                        
node    425089052                          yes      no        2   
        429740871                          yes     NaN        2   
        429951629                          yes     NaN        2   

                        operator                    ref socket:type2  \
element id                                                             
node    425089052  RWE-Effizienz  BA-0602-4 / BA-0603-0            2   
        429740871           E.ON  BA-0500-2 / BA-0501-9            2   
        429951629         Innogy  BA-0798-1 / BA-0707-5            2   

                                         source  ... socket:unknown  \
element id                                       ...                  
node    425089052  RWE Effizienz GmbH, Dortmund  ...            NaN   
        429740871                        survey  ...            NaN   
        429951629  RWE Effizienz GmbH, Dortmund  ...            NaN   

                  socket:unknown:output markings restriction street:name  \
element id                                                                 
node    425089052                   NaN      NaN         NaN         NaN   
        429740871                   NaN      NaN         NaN         NaN   
        429951629                   NaN      NaN         NaN         NaN   

                  capacity:disabled surface ref:goingelectric  \
element id                                                      
node    425089052               NaN     NaN               NaN   
        429740871               NaN     NaN               NaN   
        429951629               NaN     NaN               NaN   

                  socket:chademo:current socket:chademo:voltage  
element id                                                       
node    425089052                    NaN                    NaN  
        429740871                    NaN                    NaN  
        429951629                    NaN                    NaN  

[3 rows x 182 columns]

## 🔗 Combine Petrol & EV Charging Data

The petrol station and EV charging datasets are combined into a **single POI layer**.

To preserve semantic clarity, a new column **`poi_type`** is introduced to distinguish between
the two types of points of interest:

- `fuel_station` — traditional petrol / fuel stations  
- `charging_station` — EV charging points  

This unified structure simplifies downstream processing, analysis, and integration
with other POI layers in the database.


In [7]:
gdf_fuel["poi_type"] = "fuel_station"
gdf_ev["poi_type"] = "charging_station"

gdf = pd.concat([gdf_fuel, gdf_ev], axis=0)
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=gdf_fuel.crs)

print("Total POIs:", len(gdf))
gdf.head(3)


Total POIs: 1844


geometry amenity brand brand:wikidata  \
element id                                                                 
node    16541597   POINT (13.3456 52.54631)    fuel  Aral        Q565734   
        26756830  POINT (13.32822 52.53069)    fuel  Aral        Q565734   
        26867411   POINT (13.2945 52.50197)    fuel   NaN            NaN   

                 brand:wikipedia compressed_air fuel:HGV_diesel fuel:diesel  \
element id                                                                    
node    16541597      en:Aral AG            yes          retail         yes   
        26756830         de:Aral            yes             NaN         yes   
        26867411             NaN            yes             NaN         yes   

                 fuel:e10 fuel:octane_102  ... taxi socket:unknown  \
element id                                 ...                       
node    16541597      yes          retail  ...  NaN            NaN   
        26756830      yes             yes  ...  NaN            NaN   
        26867411      yes             NaN  ...  NaN            NaN   

                 socket:unknown:output markings restriction street:name  \
element id                                                                
node    16541597                   NaN      NaN         NaN         NaN   
        26756830                   NaN      NaN         NaN         NaN   
        26867411                   NaN      NaN         NaN         NaN   

                 capacity:disabled ref:goingelectric socket:chademo:current  \
element id                                                                    
node    16541597               NaN               NaN                    NaN   
        26756830               NaN               NaN                    NaN   
        26867411               NaN               NaN                    NaN   

                 socket:chademo:voltage  
element id                               
node    16541597                    NaN  
        26756830                    NaN  
        26867411                    NaN  

[3 rows x 278 columns]

## 1.2 Modelling & Planning

This section outlines the key modelling decisions, known data limitations,
and the planned transformation steps before database population.

---

### Selected Key Columns

- `station_name`
- `street`, `housenumber`, `postcode`, `city`
- `brand`, `operator`
- `opening_hours`
- Contact information (`phone`, `email`, `website`)
- `poi_type` (petrol station vs EV charging point)
- `latitude`, `longitude`
- `district`, `neighborhood` (derived in later steps)
- `data_source`

---

### Known Data Issues

- Address-related fields may be partially missing or incomplete
- Fuel types are inconsistently tagged in OpenStreetMap
- Contact information (phone, email, website) is often missing or outdated

---

### Planned Transformations

- Rename raw OSM columns to a standardized naming convention
- Normalize geometries and extract latitude and longitud


In [8]:
columns = [
    "name",
    "addr:street", "addr:housenumber", "addr:postcode", "addr:city",
    "opening_hours",
    "brand", "operator",
    "phone", "contact:email", "website",
    "geometry",
    "poi_type"
]

gdf_station = gdf[[c for c in columns if c in gdf.columns]].copy()
print("Filtered dataset shape:", gdf_station.shape)
gdf_station.head(3)


Filtered dataset shape: (1844, 13)


name         addr:street addr:housenumber  \
element id                                                              
node    16541597            Aral                 NaN              NaN   
        26756830            Aral       Beusselstraße               55   
        26867411  Bavaria petrol  Heilbronner Straße               14   

                 addr:postcode addr:city                      opening_hours  \
element id                                                                    
node    16541597           NaN       NaN                               24/7   
        26756830         10553    Berlin                               24/7   
        26867411         10711    Berlin  Mo-Fr 07:00-20:00; Sa 07:00-18:00   

                 brand        operator           phone contact:email  \
element id                                                             
node    16541597  Aral             NaN             NaN           NaN   
        26756830  Aral      Mike Seitz  +49 30 3914404           NaN   
        26867411   NaN  Bavaria Petrol             NaN           NaN   

                                                            website  \
element id                                                            
node    16541597                                                NaN   
        26756830  https://tankstelle.aral.de/berlin/beusselstras...   
        26867411                                                NaN   

                                   geometry      poi_type  
element id                                                 
node    16541597   POINT (13.3456 52.54631)  fuel_station  
        26756830  POINT (13.32822 52.53069)  fuel_station  
        26867411   POINT (13.2945 52.50197)  fuel_station

## 📐 Geometry Normalization & Coordinate Extraction

All geometries are normalized to **Point** objects to ensure consistency across
all records.

From the normalized geometries, **latitude** and **longitude** values are extracted.
These coordinates are required for:

- Mapping and visualization
- Spatial joins with district and neighborhood boundaries
- Integration with other geospatial layers in the platform


In [9]:
# Ensure geometry is Point
gdf_station["geometry"] = gdf_station["geometry"].apply(
    lambda geom: geom if geom.geom_type == "Point" else geom.representative_point()
)

# Extract latitude and longitude
gdf_station["latitude"] = gdf_station.geometry.y
gdf_station["longitude"] = gdf_station.geometry.x


## 🏷️ Column Normalization

OpenStreetMap-specific column names are renamed to improve readability
and to align with the standardized naming conventions used across POI layers.

In addition, a **`data_source`** column is added to each record to ensure
traceability and transparency regarding the origin of the data.


In [10]:
gdf_station = gdf_station.rename(columns={
    "name": "station_name",
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city",
    "contact:email": "email"
})

gdf_station["data_source"] = "OSM"
gdf_station.head(3)


station_name              street housenumber postcode  \
element id                                                                  
node    16541597            Aral                 NaN         NaN      NaN   
        26756830            Aral       Beusselstraße          55    10553   
        26867411  Bavaria petrol  Heilbronner Straße          14    10711   

                    city                      opening_hours brand  \
element id                                                          
node    16541597     NaN                               24/7  Aral   
        26756830  Berlin                               24/7  Aral   
        26867411  Berlin  Mo-Fr 07:00-20:00; Sa 07:00-18:00   NaN   

                        operator           phone email  \
element id                                               
node    16541597             NaN             NaN   NaN   
        26756830      Mike Seitz  +49 30 3914404   NaN   
        26867411  Bavaria Petrol             NaN   NaN   

                                                            website  \
element id                                                            
node    16541597                                                NaN   
        26756830  https://tankstelle.aral.de/berlin/beusselstras...   
        26867411                                                NaN   

                                   geometry      poi_type   latitude  \
element id                                                             
node    16541597   POINT (13.3456 52.54631)  fuel_station  52.546315   
        26756830  POINT (13.32822 52.53069)  fuel_station  52.530694   
        26867411   POINT (13.2945 52.50197)  fuel_station  52.501974   

                  longitude data_source  
element id                               
node    16541597  13.345599         OSM  
        26756830  13.328225         OSM  
        26867411  13.294496         OSM

## Step 1 Review & Data Familiarization (A–F)

This section provides an initial understanding of the raw **petrol station**
and **EV charging** dataset, including its structure, completeness, and
basic sanity checks prior to further processing.


In [11]:
print("Rows, Columns:", gdf_station.shape)
print("\nColumns:")
print(gdf_station.columns.tolist())
print("\nData Info:")
gdf_station.info()


Rows, Columns: (1844, 16)

Columns:
['station_name', 'street', 'housenumber', 'postcode', 'city', 'opening_hours', 'brand', 'operator', 'phone', 'email', 'website', 'geometry', 'poi_type', 'latitude', 'longitude', 'data_source']

Data Info:
<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 1844 entries, ('node', np.int64(16541597)) to ('way', np.int64(1446480594))
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   station_name   513 non-null    object  
 1   street         284 non-null    object  
 2   housenumber    282 non-null    object  
 3   postcode       273 non-null    object  
 4   city           267 non-null    object  
 5   opening_hours  509 non-null    object  
 6   brand          567 non-null    object  
 7   operator       1353 non-null   object  
 8   phone          105 non-null    object  
 9   email          2 non-null      object  
 10  website        160 non-null    object  
 11  ge

In [12]:
##Missing Values per column
missing_count = gdf_station.isna().sum().sort_values(ascending=False)
row_count = len(gdf_station)

missing = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": (missing_count / row_count * 100).round(1)
})

missing


,missing_count,missing_pct
email,1842,99.9
phone,1739,94.3
website,1684,91.3
city,1577,85.5
postcode,1571,85.2
housenumber,1562,84.7
street,1560,84.6
opening_hours,1335,72.4
station_name,1331,72.2
brand,1277,69.3


### ⚠️ Missing Data Summary

- Address-related fields (`street`, `housenumber`, `postcode`, `city`) are partially missing.
- Contact information (`phone`, `email`, `website`) is highly incomplete.
- Brand and operator information is present for many stations but not guaranteed.
- Geometry and coordinate values are complete for all records.


In [13]:
gdf_station.nunique().sort_values(ascending=False)

geometry         1844
longitude        1842
latitude         1839
operator          226
street            208
housenumber       199
postcode          130
station_name      117
website           117
phone              94
brand              72
opening_hours      70
email               2
poi_type            2
city                1
data_source         1
dtype: int64

In [14]:
print("POI types:")
print(gdf_station["poi_type"].value_counts())

if "brand" in gdf_station.columns:
    print("\nTop 10 brands:")
    print(gdf_station["brand"].value_counts().head(10))

if "operator" in gdf_station.columns:
    print("\nTop 10 operators:")
    print(gdf_station["operator"].value_counts().head(10))


POI types:
poi_type
charging_station    1555
fuel_station         289
Name: count, dtype: int64

Top 10 brands:
brand
ubitricity       116
Shell             57
Aral              51
TotalEnergies     38
Innogy            34
star              28
JET               22
Esso              21
Sprint            19
HEM               16
Name: count, dtype: int64

Top 10 operators:
operator
Allego                      185
Berliner Stadtwerke         179
ubitricity                  136
Qwello                       96
be emobil                    60
EZE                          46
Lidl                         40
Shell Recharge Solutions     40
Innogy                       35
RWE-Effizienz                35
Name: count, dtype: int64


##geometry sanity checks
print("Geometry types:")
print(gdf_station.geometry.geom_type.value_counts())

print("\nMissing geometries:", gdf_station.geometry.isna().sum())


In [15]:
##latitude/longitude Range check
print("Latitude range:",
      gdf_station["latitude"].min(), "to", gdf_station["latitude"].max())

print("Longitude range:",
      gdf_station["longitude"].min(), "to", gdf_station["longitude"].max())


Latitude range: 52.3756133 to 52.64092
Longitude range: 13.1397833 to 13.6959591


### ✅ Data Quality Conclusions

- All petrol stations and EV charging points have valid geometries and coordinates.
- Spatial data is reliable and suitable for mapping and spatial joins.
- Address and contact metadata is incomplete and should be treated as optional.
- Fuel types and amenities are inconsistently tagged in OpenStreetMap.


## 1.3 Schema Definition

All tables in this pipeline follow a standardized schema to ensure
compatibility with the unified POI model.

### Planned Table: `gas_stations_table`

| Column | Data Type | Description |
|------|----------|-------------|
| id | VARCHAR(20) | Primary key (numeric only) |
| district_id | VARCHAR(20) | FK mapped from district name |
| name | VARCHAR(200) | Station name (defaults to 'Unknown') |
| latitude | DECIMAL(9,6) | Latitude |
| longitude | DECIMAL(9,6) | Longitude |
| geometry | VARCHAR | POINT(lon lat) |
| neighborhood | VARCHAR(100) | Berlin neighborhood (Kiez) |
| district | VARCHAR(100) | Berlin district |
| neighborhood_id | VARCHAR(20) | FK to neighborhoods |
| fuel_types | VARCHAR(200) | Petrol, Diesel, LPG, EV charging |
| operating_hours | VARCHAR(50) | Opening hours |
| amenities | VARCHAR(200) | Shop, car wash, toilets, etc. |
| contact_info | VARCHAR(200) | Phone, email, website |
| data_source | VARCHAR(100) | Data origin (OSM, Berlin Open Data) |

### Constraint
- `district_id_fk`  
  FOREIGN KEY (`district_id`) REFERENCES `berlin_data.districts(district_id)`  
  ON DELETE RESTRICT ON UPDATE CASCADE


## 1.5 Review

✅ Data sources identified (OSM, Berlin Open Data, Wikidata)  
✅ Petrol stations and EV charging points fetched from OpenStreetMap  
✅ Relevant columns selected and normalized  
✅ Data quality assessed  
✅ Standardized schema drafted  

**Step 1 – Research & Data Modelling complete.**
